## 1. Setup

In [ ]:
import glob, os, shutil

WORK_DIR = "/kaggle/working/sentiment_analysis"
if not os.path.exists(WORK_DIR):
    found = glob.glob("/kaggle/input/**/inference.py", recursive=True)
    assert found, "inference.py not found in /kaggle/input: attach the updated repo"
    shutil.copytree(os.path.dirname(found[0]), WORK_DIR)
os.chdir(WORK_DIR)

configs = [p for p in glob.glob("/kaggle/input/**/config.json", recursive=True)
           if os.path.exists(os.path.join(os.path.dirname(p), "model.safetensors"))]
assert configs, "No folder with config.json + model.safetensors in /kaggle/input: attach the fine-tuned model"
MODEL_DIR = os.path.dirname(configs[0])
print("repo:", os.getcwd())
print("model:", MODEL_DIR, sorted(os.listdir(MODEL_DIR)))

In [ ]:
!pip install -q onnx onnxruntime openai

import json, time
import numpy as np
import pandas as pd
import torch
import transformers
import onnxruntime
from transformers import AutoTokenizer, AutoModelForSequenceClassification

import inference
from model_training import load_test_data, summarize, LABELS

print("torch", torch.__version__, "| transformers", transformers.__version__, "| onnxruntime", onnxruntime.__version__)

## 2. Export to ONNX (fp32)

In [ ]:
os.makedirs("onnx", exist_ok=True)
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR, attn_implementation="eager").eval()
assert [model.config.id2label[i] for i in range(3)] == LABELS, model.config.id2label


class Logits(torch.nn.Module):
    def __init__(self, m):
        super().__init__()
        self.m = m

    def forward(self, input_ids, attention_mask):
        return self.m(input_ids=input_ids, attention_mask=attention_mask).logits


test = load_test_data()
texts = test["text"].tolist()
sample = tokenizer(["warm up", "a slightly longer example sentence"], padding=True, return_tensors="pt")
args = (sample["input_ids"], sample["attention_mask"])
names = dict(input_names=["input_ids", "attention_mask"], output_names=["logits"])

enc = tokenizer(texts[:64], truncation=True, max_length=inference.MAX_LENGTH, padding=True, return_tensors="pt")
with torch.no_grad():
    torch_logits = model(**enc).logits.numpy()


def export_and_check(dynamo: bool) -> float:
    if dynamo:
        dyn = torch.export.Dim.DYNAMIC
        torch.onnx.export(Logits(model), args, "onnx/model.onnx", dynamo=True, external_data=False,
                          dynamic_shapes={"input_ids": {0: dyn, 1: dyn}, "attention_mask": {0: dyn, 1: dyn}}, **names)
    else:
        torch.onnx.export(Logits(model), args, "onnx/model.onnx", opset_version=17, dynamo=False,
                          dynamic_axes={"input_ids": {0: "batch", 1: "seq"}, "attention_mask": {0: "batch", 1: "seq"},
                                        "logits": {0: "batch"}}, **names)
    session = onnxruntime.InferenceSession("onnx/model.onnx", providers=["CPUExecutionProvider"])
    onnx_logits = session.run(None, {"input_ids": enc["input_ids"].numpy(), "attention_mask": enc["attention_mask"].numpy()})[0]
    return float(np.abs(torch_logits - onnx_logits).max())


diff = float("inf")
for dynamo in (True, False):
    name = "dynamo" if dynamo else "TorchScript"
    try:
        diff = export_and_check(dynamo)
    except Exception as e:
        print(f"{name} exporter failed: {type(e).__name__}: {e}")
        continue
    print(f"{name} exporter: max |torch - onnx| logit difference {diff:.2e} on 64 padded tweets")
    if diff < 1e-3:
        break
assert diff < 1e-3, "No exporter reproduced PyTorch's logits. Send this log."
print(f"onnx/model.onnx: {os.path.getsize('onnx/model.onnx') / 1e6:.0f} MB")

## 3. Save the tokenizer

In [ ]:
from onnxruntime.quantization import QuantType, quantize_dynamic

quantize_dynamic("onnx/model.onnx", inference.ONNX_FILE, weight_type=QuantType.QInt8)
tokenizer.save_pretrained("onnx") 
for f in ["onnx/model.onnx", inference.ONNX_FILE, inference.TOKENIZER_FILE]:
    print(f"{f}: {os.path.getsize(f) / 1e6:.1f} MB")

## 4. Tokenizer check

In [ ]:
tok, session_int8 = inference._load_onnx(inference.TOKENIZER_FILE, inference.ONNX_FILE)
mismatch = 0
for i in range(0, len(texts), 50):
    batch = texts[i:i + 50]
    ref = tokenizer(batch, truncation=True, max_length=inference.MAX_LENGTH, padding=True)["input_ids"]
    mismatch += sum(e.ids != r for e, r in zip(tok.encode_batch(batch), ref))
print("tokenizer mismatches:", mismatch)
assert mismatch == 0

## 5. Evaluate the deployed setup

In [ ]:
inference._distilroberta = lambda: (tok, session_int8)

preds = []
for i in range(0, len(texts), inference.MAX_SENTENCES):
    preds += [r["sentiment"] for r in inference.predict(texts[i:i + inference.MAX_SENTENCES], "distilroberta")]
test["pred"] = preds
metrics = summarize(test["sentiment"], test["pred"])

fp32 = pd.read_csv("results/distilroberta_test_predictions.csv").set_index("id").loc[test["id"]]
fp32_metrics = summarize(fp32["sentiment"], fp32["pred"])
agreement = float((fp32["pred"].values == test["pred"].values).mean())

times = []
for t in texts:
    start = time.perf_counter()
    inference.predict([t], "distilroberta")
    times.append((time.perf_counter() - start) * 1000)

metrics.update(latency_ms=float(np.mean(times)), latency_ms_median=float(np.median(times)),
               agreement_with_fp32=agreement, model_size_mb=os.path.getsize(inference.ONNX_FILE) / 1e6,
               model="distilroberta ONNX int8", test_size=len(test))
print(pd.DataFrame({"ONNX int8 (deployed)": metrics, "PyTorch fp32": fp32_metrics}).loc[
    ["accuracy", "precision_macro", "recall_macro", "f1_macro"]].round(4))
print(f"\nagreement with fp32: {agreement:.1%} | latency median {metrics['latency_ms_median']:.1f} ms "
      f"(1 thread) | model {metrics['model_size_mb']:.0f} MB")

## 6. Save results

In [ ]:
os.makedirs("results", exist_ok=True)
json.dump(metrics, open("results/distilroberta_onnx_int8_metrics.json", "w"), indent=2)
test[["id", "text", "sentiment", "pred"]].to_csv("results/distilroberta_onnx_int8_test_predictions.csv", index=False)
!zip -qj onnx_results.zip results/distilroberta_onnx_int8_metrics.json results/distilroberta_onnx_int8_test_predictions.csv
print("Download from the Output panel:", os.path.abspath("onnx_results.zip"))

## 7. Upload to HF

In [ ]:
HUB_REPO = "dionsylvester/distilroberta-tweet-sentiment"

if HUB_REPO:
    from huggingface_hub import HfApi
    from kaggle_secrets import UserSecretsClient

    api = HfApi(token=UserSecretsClient().get_secret("HF_TOKEN"))
    api.create_repo(HUB_REPO, exist_ok=True)
    api.upload_folder(repo_id=HUB_REPO, folder_path=MODEL_DIR, commit_message="Fine-tuned DistilRoBERTa (fp32)")
    api.upload_folder(repo_id=HUB_REPO, folder_path="onnx", path_in_repo="onnx",
                      allow_patterns=["model.int8.onnx", "tokenizer.json"], commit_message="ONNX int8 for the API")
    print(f"Uploaded: https://huggingface.co/{HUB_REPO}")
else:
    print("HUB_REPO is empty: skipped upload")